# Week 4 — EDA: DOTA v1.0 Subset

**Goal:** select 200 tiles (160 train / 40 test, `random.seed(7)`), visualise images with oriented bounding-box annotations, analyse class distribution, and export a sample grid to `outputs/`.

> Run `python scripts/download_dota.py` first if you haven't downloaded the data yet.

In [ ]:
import sys
from pathlib import Path

# Resolve project root: walk up from cwd until we find README.md
def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'README.md').exists() and (p / 'src').exists():
            return p
    raise RuntimeError(f"Could not find project root from {start}")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
import cv2

from src.dota_utils import (
    DOTA_CLASSES, CLASS_COLORS,
    load_dota_split, select_subset, train_test_split,
    draw_annotations, make_sample_grid,
    materialize_split, write_dota_yaml, write_manifest,
)

DATA_ROOT  = PROJECT_ROOT / 'data' / 'raw' / 'DOTA'
CLEAN_ROOT = PROJECT_ROOT / 'data' / 'clean'
OUTPUTS    = PROJECT_ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')

## 1. Load dataset

In [ ]:
train_samples = load_dota_split(
    DATA_ROOT / 'train' / 'images',
    DATA_ROOT / 'train' / 'labelTxt' / 'labelTxt',
)
val_samples = load_dota_split(
    DATA_ROOT / 'val' / 'images',
    DATA_ROOT / 'val' / 'labelTxt' / 'labelTxt',
)
all_samples = train_samples + val_samples
print(f'Total tiles found: {len(all_samples)} (train={len(train_samples)}, val={len(val_samples)})')

## 2. Select 200-source subset and 160/40 train/test split (seed=7)

In [ ]:
source_subset = select_subset(all_samples, n=200, seed=7)
source_train, source_test = train_test_split(source_subset, n_train=160, seed=7)
print(f'Source subset: {len(source_subset)} images  →  train={len(source_train)}, test={len(source_test)}')

## 4. Class distribution (on tiled subset)

After tiling, some annotations from the source images fall outside the tile and are dropped, so this distribution may differ from a global DOTA distribution.

In [ ]:
train_set = materialize_split(source_train, CLEAN_ROOT, split='train', tile_size=1024)
test_set  = materialize_split(source_test,  CLEAN_ROOT, split='test',  tile_size=1024)
subset    = train_set + test_set   # used by downstream cells

write_dota_yaml(CLEAN_ROOT / 'dota.yaml', CLEAN_ROOT)
write_manifest(
    {'train': train_set, 'test': test_set},
    OUTPUTS / 'manifest.json',
    seed=7, tile_size=1024,
)
print(f'Materialized {len(subset)} tiles → {CLEAN_ROOT}')
print(f'Manifest: {OUTPUTS / "manifest.json"}')
print(f'YOLO config: {CLEAN_ROOT / "dota.yaml"}')

## 5. Sample annotated image (single tile)

In [ ]:
counter = Counter()
for s in subset:
    for ann in s.annotations:
        counter[ann.category] += 1

classes  = [c for c in DOTA_CLASSES if c in counter]
counts   = [counter[c] for c in classes]
colors   = [tuple(v/255 for v in CLASS_COLORS.get(c, (180,180,180))) for c in classes]

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(classes, counts, color=colors, edgecolor='white', linewidth=0.5)
ax.set_title('Instance count per class — 200-tile subset', fontsize=13)
ax.set_xlabel('Class')
ax.set_ylabel('Instances')
plt.xticks(rotation=45, ha='right')
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(cnt), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUTS / 'class_distribution.png', dpi=150)
plt.show()
print(f'Total annotations: {sum(counts)}')

## 6. 4×4 annotated sample grid → `outputs/sample_grid.png`

In [ ]:
sample = subset[0]
img_bgr = cv2.imread(str(sample.image_path))
img_ann = draw_annotations(img_bgr, sample.annotations)
img_rgb = cv2.cvtColor(img_ann, cv2.COLOR_BGR2RGB)

legend = [
    mpatches.Patch(color=tuple(v/255 for v in CLASS_COLORS[c]), label=c)
    for c in DOTA_CLASSES if any(ann.category == c for ann in sample.annotations)
]
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(img_rgb)
ax.axis('off')
ax.set_title(sample.name)
ax.legend(handles=legend, loc='upper right', fontsize=7, framealpha=0.7)
plt.tight_layout()
plt.show()

## 7. Basic image statistics (sampled)

In [ ]:
grid = make_sample_grid(subset, n_cols=4, tile_size=512, max_tiles=16)

fig, ax = plt.subplots(figsize=(16, 16))
ax.imshow(grid)
ax.axis('off')
ax.set_title('DOTA v1.0 — 16 sample tiles with OBB annotations', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUTS / 'sample_grid.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved to outputs/sample_grid.png')

## 8. Annotations-per-tile distribution

In [ ]:
import pandas as pd

rows = []
for s in subset[:20]:  # quick sample of 20
    img = cv2.imread(str(s.image_path))
    if img is None:
        continue
    h, w = img.shape[:2]
    rows.append({
        'name':        s.name,
        'width':       w,
        'height':      h,
        'n_instances': len(s.annotations),
        'classes':     len({a.category for a in s.annotations}),
        'mean_brightness': float(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).mean()),
    })

df = pd.DataFrame(rows)
display(df.describe().round(1))

## 7. Annotations-per-image distribution

In [ ]:
ann_counts = [len(s.annotations) for s in subset]

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(ann_counts, bins=30, color='steelblue', edgecolor='white')
ax.set_title('Annotations per tile — 200-tile subset')
ax.set_xlabel('# annotations')
ax.set_ylabel('# tiles')
ax.axvline(np.mean(ann_counts), color='red', linestyle='--', label=f'mean={np.mean(ann_counts):.1f}')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUTS / 'ann_per_tile.png', dpi=120)
plt.show()